# Flood-It NFQ Explorer — Faster Training Config

Cleaned notebook for training a small **4×4 / 3-color Flood-It** agent with a simple NFQ/DQN-style Q-network.

This version removes the old smoke-test training cell and uses faster training defaults:

- Evaluation during training happens every `EVAL_EVERY` episodes instead of every episode.
- Evaluation uses only `EVAL_EPISODES` episodes during training.
- Final evaluation is configurable through `FINAL_EVAL_EPISODES`.
- Printing happens every `PRINT_EVERY` episodes.
- `evaluate()` has a `max_steps` safety cap.
- Experience batches are built explicitly.
- States are flattened before entering the fully connected Q-network.


In [31]:
import warnings
warnings.filterwarnings("ignore")

import os
import gc
import time
import random
import tempfile
from itertools import count

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from floodit_env import FloodItEnv

print("Python/PyTorch setup")
print("torch:", torch.__version__)
print("cuda/rocm available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))


Python/PyTorch setup
torch: 2.9.1+rocm7.2.1
cuda/rocm available: True
device: AMD Radeon RX 9070 XT


## Configuration

For quick iteration, start with one seed. Once learning looks stable, increase `SEEDS`, `MAX_EPISODES`, and `MAX_MINUTES`.


In [32]:
# Seeds
SEEDS = [0]

# Training limits
MAX_MINUTES = 15
MAX_EPISODES = 5000
GOAL_MEAN_100_REWARD = 90

# Evaluation/logging schedule
EVAL_EVERY = 50
EVAL_EPISODES = 30
FINAL_EVAL_EPISODES = 100
PRINT_EVERY = 50

# NFQ / model hyperparameters
GAMMA = 0.99
HIDDEN_DIMS = (32, 32)
LEARNING_RATE = 3e-4
EPSILON = 0.1
BATCH_SIZE = 128
EPOCHS = 1

## Quick environment check

This verifies that the custom environment can be created, reset, and stepped before we train the agent.


In [33]:
env = FloodItEnv()
observation, info = env.reset(seed=0)

print("Observation space:", env.observation_space)
print("Action space:", env.action_space)
print("Initial observation shape:", observation.shape)
print("Initial board:")
print(observation)
print("Info:", info)

action = env.action_space.sample()
next_observation, reward, terminated, truncated, info = env.step(action)

print("Sample action:", action)
print("Next observation shape:", next_observation.shape)
print("Reward:", reward)
print("Terminated:", terminated, "Truncated:", truncated)
print("Info:", info)

env.close()


Observation space: Box(0, 2, (4, 4), int64)
Action space: Discrete(3)
Initial observation shape: (4, 4)
Initial board:
[[1 1 0 1]
 [2 1 1 1]
 [1 1 2 0]
 [2 0 1 0]]
Info: {'moves': 0, 'moves_left': 25, 'solved': False, 'action_mask': array([1, 0, 1], dtype=int8)}
Sample action: 2
Next observation shape: (4, 4)
Reward: 2.0
Terminated: False Truncated: False
Info: {'moves': 1, 'moves_left': 24, 'solved': False, 'action_mask': array([1, 1, 0], dtype=int8)}


## Fully connected Q-network

`FCQ` receives a flattened board state and outputs one Q-value per possible color/action.

For a 4×4 board with 3 colors:

```text
input_dim = 16
output_dim = 3
network = 16 → 32 → 32 → 3
```


In [34]:
def encode_state(state, n_colors):
    """Represent each color as a categorical one-hot vector instead of an ordinal id."""
    arr = np.asarray(state)

    # Already encoded: [cells * colors].
    if arr.ndim == 1 and arr.size % n_colors == 0:
        cells = arr.reshape(-1, n_colors)
        if np.all((cells == 0) | (cells == 1)) and np.all(cells.sum(axis=1) == 1):
            return arr.astype(np.float32, copy=False)

    color_ids = arr.astype(np.int64, copy=False).reshape(-1)
    return np.eye(n_colors, dtype=np.float32)[color_ids].reshape(-1)


class FCQ(nn.Module):
    def __init__(self,
                 input_dim,
                 output_dim,
                 hidden_dims=(32, 32),
                 activation_fc=F.relu):
        super().__init__()
        self.activation_fc = activation_fc

        self.input_layer = nn.Linear(input_dim, hidden_dims[0])

        self.hidden_layers = nn.ModuleList()
        for i in range(len(hidden_dims) - 1):
            self.hidden_layers.append(nn.Linear(hidden_dims[i], hidden_dims[i + 1]))

        self.output_layer = nn.Linear(hidden_dims[-1], output_dim)

        self.input_dim = input_dim
        self.output_dim = output_dim
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        self.to(self.device)

    def _format(self, state):
        """Convert raw color-id boards or one-hot states into model-ready tensors."""
        if not isinstance(state, torch.Tensor):
            state = torch.tensor(state, dtype=torch.float32)

        state = state.to(self.device).float()
        expected_raw_cells = self.input_dim // self.output_dim

        if state.dim() == 1:
            # Single flattened state. Accept either already one-hot [input_dim]
            # or raw color ids [cells], which we encode here.
            if state.numel() == expected_raw_cells:
                state = F.one_hot(state.long(), num_classes=self.output_dim).float().flatten().unsqueeze(0)
            else:
                state = state.unsqueeze(0)

        elif state.dim() == 2:
            if state.shape[1] == self.input_dim:
                # Batch of already encoded states: [B, input_dim].
                pass
            elif state.shape[1] == expected_raw_cells:
                # Batch of raw flattened boards: [B, cells] -> [B, cells*num_colors].
                state = F.one_hot(state.long(), num_classes=self.output_dim).float().reshape(state.shape[0], -1)
            elif state.numel() == expected_raw_cells:
                # Single raw board: [H, W] -> [1, cells*num_colors].
                state = F.one_hot(state.long().flatten(), num_classes=self.output_dim).float().flatten().unsqueeze(0)
            else:
                raise ValueError(f"Expected state width {self.input_dim} or raw width {expected_raw_cells}, got shape {tuple(state.shape)}")

        if state.shape[-1] != self.input_dim:
            raise ValueError(f"Expected state width {self.input_dim}, got {state.shape[-1]}")

        return state

    def forward(self, state):
        x = self._format(state)
        x = self.activation_fc(self.input_layer(x))

        for hidden_layer in self.hidden_layers:
            x = self.activation_fc(hidden_layer(x))

        return self.output_layer(x)

    def load(self, experiences):
        """Convert NumPy experience batches to tensors on the correct device."""
        states, actions, rewards, next_states, is_terminals = experiences

        states = torch.from_numpy(states).float().to(self.device)
        actions = torch.from_numpy(actions).long().to(self.device)
        rewards = torch.from_numpy(rewards).float().to(self.device)
        next_states = torch.from_numpy(next_states).float().to(self.device)
        is_terminals = torch.from_numpy(is_terminals).float().to(self.device)

        return states, actions, rewards, next_states, is_terminals


## Action selection strategies

- `GreedyStrategy`: always choose the action with the highest predicted Q-value.
- `EGreedyStrategy`: usually choose the best predicted action, but sometimes choose a random action for exploration.


In [35]:
def current_color_from_state(state, n_actions):
    arr = np.asarray(state).reshape(-1)

    # One-hot board states store the top-left cell in the first n_actions entries.
    if arr.size % n_actions == 0 and arr.size > n_actions:
        first_cell = arr[:n_actions]
        if np.all((first_cell == 0) | (first_cell == 1)) and first_cell.sum() == 1:
            return int(np.argmax(first_cell))

    return int(arr[0])


def valid_color_actions(state, n_actions):
    current_color = current_color_from_state(state, n_actions)
    return [action for action in range(n_actions) if action != current_color]


class GreedyStrategy:
    def __init__(self):
        self.exploratory_action_taken = False

    def select_action(self, model, state):
        with torch.no_grad():
            q_values = model(state).cpu().numpy().squeeze()

        valid_actions = valid_color_actions(state, len(q_values))
        masked_q_values = q_values.copy()
        masked_q_values[:] = -np.inf
        masked_q_values[valid_actions] = q_values[valid_actions]
        return int(np.argmax(masked_q_values))


class EGreedyStrategy:
    def __init__(self, epsilon=0.1):
        self.epsilon = epsilon
        self.exploratory_action_taken = None

    def select_action(self, model, state):
        with torch.no_grad():
            q_values = model(state).cpu().numpy().squeeze()

        valid_actions = valid_color_actions(state, len(q_values))
        masked_q_values = q_values.copy()
        masked_q_values[:] = -np.inf
        masked_q_values[valid_actions] = q_values[valid_actions]
        greedy_action = int(np.argmax(masked_q_values))

        if np.random.rand() > self.epsilon:
            self.exploratory_action_taken = False
            return greedy_action

        self.exploratory_action_taken = True
        return int(np.random.choice(valid_actions))


## NFQ trainer

This trainer:

1. Plays episodes using epsilon-greedy exploration.
2. Stores `(state, action, reward, next_state, done)` experiences.
3. Once enough experiences are collected, trains the Q-network for several epochs on that batch.
4. Evaluates periodically with a greedy policy.


In [36]:
REPRESENTATION_VERSION = "one_hot_actions_masked_targets_v2"


class NFQ:
    def __init__(self,
                 value_model_fn,
                 value_optimizer_fn,
                 value_optimizer_lr,
                 training_strategy_fn,
                 evaluation_strategy_fn,
                 batch_size,
                 epochs,
                 eval_every=25,
                 eval_episodes=3,
                 final_eval_episodes=10,
                 print_every=25,
                 max_eval_steps=1000):
        self.value_model_fn = value_model_fn
        self.value_optimizer_fn = value_optimizer_fn
        self.value_optimizer_lr = value_optimizer_lr
        self.training_strategy_fn = training_strategy_fn
        self.evaluation_strategy_fn = evaluation_strategy_fn
        self.batch_size = batch_size
        self.epochs = epochs
        self.eval_every = eval_every
        self.eval_episodes = eval_episodes
        self.final_eval_episodes = final_eval_episodes
        self.print_every = print_every
        self.max_eval_steps = max_eval_steps

    def _masked_action_values(self, q_values, states):
        masked_q_values = q_values.clone()

        # States are one-hot encoded as [cell0_color0, cell0_color1, ...].
        # The first nA entries describe the top-left/current color, whose action is a no-op.
        current_colors = states[:, :q_values.shape[1]].argmax(dim=1)
        masked_q_values[torch.arange(q_values.shape[0], device=q_values.device), current_colors] = -torch.inf
        return masked_q_values

    def optimize_model(self, experiences):
        states, actions, rewards, next_states, is_terminals = experiences

        # Q-learning target:
        # target = reward + gamma * max_valid_a Q(next_state, a), unless terminal.
        next_q_values = self.online_model(next_states).detach()
        max_a_q_sp = self._masked_action_values(next_q_values, next_states).max(1)[0].unsqueeze(1)
        target_q_s = rewards + self.gamma * max_a_q_sp * (1 - is_terminals)

        # Q-value predicted for the valid action actually taken.
        q_sa = self.online_model(states).gather(1, actions)

        td_errors = q_sa - target_q_s
        value_loss = td_errors.pow(2).mul(0.5).mean()

        self.value_optimizer.zero_grad()
        value_loss.backward()
        self.value_optimizer.step()

    def interaction_step(self, state, env):
        action = self.training_strategy.select_action(self.online_model, state)

        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        next_state = encode_state(next_state, env.action_space.n)

        experience = (state, action, reward, next_state, float(done))
        self.experiences.append(experience)

        self.episode_reward[-1] += reward
        self.episode_timestep[-1] += 1
        self.episode_exploration[-1] += int(self.training_strategy.exploratory_action_taken)

        return next_state, done

    def _make_batches(self):
        states, actions, rewards, next_states, is_terminals = zip(*self.experiences)

        return (
            np.vstack(states).astype(np.float32),
            np.array(actions, dtype=np.int64).reshape(-1, 1),
            np.array(rewards, dtype=np.float32).reshape(-1, 1),
            np.vstack(next_states).astype(np.float32),
            np.array(is_terminals, dtype=np.float32).reshape(-1, 1),
        )

    def train(self, make_env_fn, make_env_kargs, seed, gamma,
              max_minutes, max_episodes, goal_mean_100_reward):
        training_start = time.time()

        self.checkpoint_dir = tempfile.mkdtemp()
        self.make_env_fn = make_env_fn
        self.make_env_kargs = make_env_kargs
        self.seed = seed
        self.gamma = gamma

        torch.manual_seed(seed)
        np.random.seed(seed)
        random.seed(seed)

        env = self.make_env_fn(**self.make_env_kargs, seed=self.seed)

        nA = env.action_space.n
        nS = int(np.prod(env.observation_space.shape)) * nA

        self.episode_timestep = []
        self.episode_reward = []
        self.episode_seconds = []
        self.evaluation_scores = []
        self.episode_exploration = []

        print(f"Representation: {REPRESENTATION_VERSION}; nS={nS}; nA={nA}", flush=True)
        self.online_model = self.value_model_fn(nS, nA)
        self.value_optimizer = self.value_optimizer_fn(self.online_model, self.value_optimizer_lr)

        self.training_strategy = self.training_strategy_fn()
        self.evaluation_strategy = self.evaluation_strategy_fn()
        self.experiences = []

        result = np.empty((max_episodes, 5))
        result[:] = np.nan
        training_time = 0.0

        for episode in range(1, max_episodes + 1):
            episode_start = time.time()

            state, info = env.reset()
            state = encode_state(state, env.action_space.n)
            done = False

            self.episode_reward.append(0.0)
            self.episode_timestep.append(0.0)
            self.episode_exploration.append(0.0)

            for step in count():
                state, done = self.interaction_step(state, env)

                if len(self.experiences) >= self.batch_size:
                    batches = self._make_batches()
                    tensor_experiences = self.online_model.load(batches)

                    for _ in range(self.epochs):
                        self.optimize_model(tensor_experiences)

                    self.experiences.clear()

                if done:
                    gc.collect()
                    break

            episode_elapsed = time.time() - episode_start
            self.episode_seconds.append(episode_elapsed)
            training_time += episode_elapsed

            wallclock_elapsed = time.time() - training_start
            reached_max_minutes = wallclock_elapsed >= max_minutes * 60
            reached_max_episodes = episode >= max_episodes

            # Evaluate less often to keep training fast.
            should_evaluate = episode % self.eval_every == 0 or reached_max_minutes or reached_max_episodes
            if should_evaluate:
                evaluation_score, _ = self.evaluate(
                    self.online_model,
                    env,
                    n_episodes=self.eval_episodes,
                    max_steps=self.max_eval_steps,
                )
            else:
                evaluation_score = np.nan
            self.evaluation_scores.append(evaluation_score)

            total_step = int(np.sum(self.episode_timestep))
            mean_10_reward = np.mean(self.episode_reward[-10:])
            std_10_reward = np.std(self.episode_reward[-10:])
            mean_100_reward = np.mean(self.episode_reward[-100:])
            std_100_reward = np.std(self.episode_reward[-100:])

            recent_eval_scores = np.array(self.evaluation_scores[-100:], dtype=np.float32)
            if np.all(np.isnan(recent_eval_scores)):
                mean_100_eval_score = float("-inf")
                std_100_eval_score = 0.0
            else:
                mean_100_eval_score = np.nanmean(recent_eval_scores)
                std_100_eval_score = np.nanstd(recent_eval_scores)

            timesteps = np.array(self.episode_timestep[-100:])
            explorations = np.array(self.episode_exploration[-100:])
            exploration_ratios = explorations / np.maximum(timesteps, 1)
            mean_100_exp_rat = np.mean(exploration_ratios)
            std_100_exp_rat = np.std(exploration_ratios)

            result[episode - 1] = (
                total_step,
                mean_100_reward,
                mean_100_eval_score,
                training_time,
                wallclock_elapsed,
            )

            reached_goal_mean_reward = mean_100_eval_score >= goal_mean_100_reward
            training_is_over = reached_max_minutes or reached_max_episodes or reached_goal_mean_reward

            elapsed_str = time.strftime("%H:%M:%S", time.gmtime(wallclock_elapsed))
            eval_text = "nan" if np.isnan(evaluation_score) else f"{evaluation_score:05.1f}"
            mean_eval_text = "nan" if mean_100_eval_score == float("-inf") else f"{mean_100_eval_score:05.1f}"
            debug_message = (
                f"el {elapsed_str}, ep {episode:04}, ts {total_step:06}, "
                f"ar10 {mean_10_reward:05.1f}±{std_10_reward:05.1f}, "
                f"ar100 {mean_100_reward:05.1f}±{std_100_reward:05.1f}, "
                f"ex100 {mean_100_exp_rat:04.2f}±{std_100_exp_rat:04.2f}, "
                f"eval {eval_text}, ev100 {mean_eval_text}±{std_100_eval_score:05.1f}"
            )

            if episode % self.print_every == 0 or training_is_over:
                print(debug_message, flush=True)

            if should_evaluate:
                self.save_checkpoint(episode - 1, self.online_model)

            if training_is_over:
                if reached_max_minutes:
                    print("--> reached_max_minutes ✕")
                if reached_max_episodes:
                    print("--> reached_max_episodes ✕")
                if reached_goal_mean_reward:
                    print("--> reached_goal_mean_reward ✓")
                break

        final_eval_score, score_std = self.evaluate(
            self.online_model,
            env,
            n_episodes=self.final_eval_episodes,
            max_steps=self.max_eval_steps,
        )
        wallclock_time = time.time() - training_start

        print("Training complete.")
        print(
            f"Final evaluation score {final_eval_score:.2f}±{score_std:.2f} "
            f"in {training_time:.2f}s training time, {wallclock_time:.2f}s wall-clock time."
        )

        env.close()
        return result, final_eval_score, training_time, wallclock_time

    def evaluate(self, eval_policy_model, eval_env, n_episodes=1, max_steps=100):
        rewards = []

        for _ in range(n_episodes):
            state, info = eval_env.reset()
            state = encode_state(state, eval_env.action_space.n)
            rewards.append(0.0)

            for _ in range(max_steps):
                action = self.evaluation_strategy.select_action(eval_policy_model, state)
                state, reward, terminated, truncated, info = eval_env.step(action)
                done = terminated or truncated
                state = encode_state(state, eval_env.action_space.n)
                rewards[-1] += reward

                if done:
                    break

        return np.mean(rewards), np.std(rewards)

    def save_checkpoint(self, episode_idx, model):
        path = os.path.join(self.checkpoint_dir, f"model.{episode_idx}.tar")
        torch.save(model.state_dict(), path)


## Environment factory

The trainer expects a function that accepts `seed=` and returns an environment. We seed once on creation, then the trainer resets normally each episode.


In [37]:
def make_floodit_env(seed=None):
    env = FloodItEnv()
    env.reset(seed=seed)
    return env

make_env_fn = make_floodit_env
make_env_kargs = {}


## Training

This is the main training cell. It uses the faster evaluation schedule configured above.

To run a longer experiment, increase `SEEDS`, `MAX_EPISODES`, and `MAX_MINUTES` in the configuration cell.


In [38]:
nfq_results = []
best_agent, best_eval_score = None, float("-inf")

for seed in SEEDS:
    print(f"\n=== Training seed {seed} ===")

    value_model_fn = lambda nS, nA: FCQ(
        input_dim=nS,
        output_dim=nA,
        hidden_dims=HIDDEN_DIMS,
    )

    value_optimizer_fn = lambda net, lr: optim.Adam(net.parameters(), lr=lr)
    training_strategy_fn = lambda: EGreedyStrategy(epsilon=EPSILON)
    evaluation_strategy_fn = lambda: GreedyStrategy()

    agent = NFQ(
        value_model_fn=value_model_fn,
        value_optimizer_fn=value_optimizer_fn,
        value_optimizer_lr=LEARNING_RATE,
        training_strategy_fn=training_strategy_fn,
        evaluation_strategy_fn=evaluation_strategy_fn,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        eval_every=EVAL_EVERY,
        eval_episodes=EVAL_EPISODES,
        final_eval_episodes=FINAL_EVAL_EPISODES,
        print_every=PRINT_EVERY,
        max_eval_steps=100,
    )

    result, final_eval_score, training_time, wallclock_time = agent.train(
        make_env_fn=make_env_fn,
        make_env_kargs=make_env_kargs,
        seed=seed,
        gamma=GAMMA,
        max_minutes=MAX_MINUTES,
        max_episodes=MAX_EPISODES,
        goal_mean_100_reward=GOAL_MEAN_100_REWARD,
    )

    nfq_results.append(result)

    if final_eval_score > best_eval_score:
        best_eval_score = final_eval_score
        best_agent = agent

nfq_results = np.array(nfq_results)
print("\nBest eval score:", best_eval_score)



=== Training seed 0 ===
Representation: one_hot_actions_masked_targets_v2; nS=48; nA=3
el 00:00:04, ep 0050, ts 000676, ar10 077.7±048.7, ar100 072.2±053.6, ex100 0.13±0.13, eval 017.1, ev100 017.1±000.0
el 00:00:09, ep 0100, ts 001351, ar10 059.0±058.1, ar100 072.6±052.6, ex100 0.13±0.13, eval 006.6, ev100 011.9±005.2
el 00:00:14, ep 0150, ts 002039, ar10 066.9±055.8, ar100 073.5±051.8, ex100 0.13±0.12, eval 018.9, ev100 012.8±006.1
el 00:00:19, ep 0200, ts 002748, ar10 066.7±053.4, ar100 068.8±054.8, ex100 0.11±0.11, eval 003.1, ev100 011.0±007.9
el 00:00:23, ep 0250, ts 003466, ar10 066.5±053.6, ar100 069.5±053.7, ex100 0.11±0.11, eval 016.2, ev100 009.6±006.5
el 00:00:28, ep 0300, ts 004195, ar10 080.7±047.8, ar100 070.3±053.3, ex100 0.13±0.11, eval 016.4, ev100 016.3±000.1
el 00:00:33, ep 0350, ts 004956, ar10 056.2±061.3, ar100 063.9±056.7, ex100 0.13±0.11, eval 021.0, ev100 018.7±002.3
el 00:00:37, ep 0400, ts 005591, ar10 091.0±037.2, ar100 069.2±054.8, ex100 0.13±0.11, eval 0

## Inspect the trained policy

This runs one greedy episode with the best agent and prints each board.


In [40]:
def play_one_episode(agent, seed=123):
    env = FloodItEnv()
    state, info = env.reset(seed=seed)
    done = False
    total_reward = 0.0
    step = 0

    print("Initial board:")
    print(state)

    while not done:
        flat_state = encode_state(state, env.action_space.n)
        action = agent.evaluation_strategy.select_action(agent.online_model, flat_state)
        state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        total_reward += reward
        step += 1

        print(f"\nStep {step}, action={action}, reward={reward:.2f}")
        print(state)

    print("\nDone")
    print("Total reward:", total_reward)
    print("Info:", info)
    env.close()


# Run after training:
# play_one_episode(best_agent)


In [ ]:
play_one_episode(best_agent,seed=0)